# Configurable Galaxy Zoo experiments
Edit `configs/experiments.toml` for model, profile, architecture, device, and seed selection. This notebook uses the same APIs as the CLI. Prepare/download the raw data once before running. The training cell trains and evaluates only the configured model; the benchmark cell is separate and opt-in.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from qmla.config import load_config
from qmla.experiments import run_experiments

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs' / 'experiments.toml').is_file())
config = load_config(ROOT / 'configs' / 'experiments.toml')
print(f'Model: {config.run.model}; profile: {config.run.profile}; device: {config.training.device}')
print(f'Cache: {config.cache_dir}')
config.resolved_dict()

## Train and test the selected model
For quick prototyping, select `smoke` or `small_learning` in TOML before loading the configuration above. Full profiles can take substantially longer.

In [ ]:
rows = run_experiments(config)
pd.DataFrame(rows)

In [ ]:
run_dir = Path(rows[0]['run_dir'])
history = pd.read_csv(run_dir / 'history.csv')
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
history.plot(x='epoch', y=['train_loss', 'validation_loss'], ax=axes[0])
history.plot(x='epoch', y='macro_f1', ax=axes[1])
fig.tight_layout()
plt.show()
json.loads((run_dir / 'runtime.json').read_text())

## Optional benchmark
Uncomment the following call to train/test all three model families using the same configured profile, data and seeds.

In [ ]:
# comparison = run_experiments(config, benchmark=True)
# pd.DataFrame(comparison)